# AVA v3-Code — resumable training notebook (Colab / Kaggle / laptop)

**One loop, three platforms, preemption-safe.** Every platform is a disposable worker.
The canonical state lives in a Hugging Face Hub repo. Any session can die; re-run this
notebook anywhere and it resumes from the last push within one sync interval.

Pattern (from `scripts/checkpoint_sync.py`):
- checkpoint = **trainable params + optimizer + RNG + step**, pushed every `SYNC_MINUTES`
- `LATEST.json` is uploaded **last** → a torn upload never corrupts resume
- data order is deterministic from `seed + step` → we fast-forward the stream on resume
- each session is **time-boxed** to `SHARD_MINUTES` (< platform limit), then force-saves and exits

Phases (CODE_PIVOT.md §8): **C1** donor baseline → **C2** linearize → **C3** MoE/ternary →
**C4** mount HRM → **C5** SFT → **C6** self-play RL → **C7** pack → **C8** eval/release.
This notebook runs **C1 (donor eval)** and a generic **LoRA shard trainer** you point at any phase's data.

> ⚠️ Read the review doc `docs/v3/REVIEW_2026-07.md` first — it changes what C2/C3 should be.

## 0. Setup

In [ ]:
# Free-quota friendly. bitsandbytes needs a CUDA GPU (Colab/Kaggle T4 = fine).
!pip -q install -U 'transformers>=4.44' 'datasets>=2.20' 'accelerate>=0.33' \
    'peft>=0.12' 'bitsandbytes>=0.43' 'huggingface_hub>=0.24' 2>/dev/null
print('deps ready')

## 1. Auth — HF token (never hard-code it)
- **Kaggle**: Add-ons → Secrets → add `HF_TOKEN`.
- **Colab**: 🔑 sidebar → add secret `HF_TOKEN`.
- **Laptop**: `export HF_TOKEN=...` or `huggingface-cli login` once.

In [ ]:
import os
tok = os.environ.get('HF_TOKEN')
if not tok:
    try:  # Kaggle
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not tok:
    try:  # Colab
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
    except Exception:
        pass
assert tok, 'Set HF_TOKEN (secret/env). It needs WRITE on your checkpoint repo.'
os.environ['HF_TOKEN'] = tok
from huggingface_hub import login; login(tok, add_to_git_credential=False)
print('hf auth ok')

## 2. Platform + GPU

In [ ]:
import torch, platform
def where():
    if os.path.exists('/kaggle'): return 'kaggle'
    if 'COLAB_GPU' in os.environ or os.path.exists('/content'): return 'colab'
    return 'local'
PLATFORM = where()
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only'
vram = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
print(f'platform={PLATFORM}  gpu={gpu}  vram={vram:.1f}GB  torch={torch.__version__}')
# Time-box below platform hard limits (leave margin for the final push).
SHARD_MINUTES = {'kaggle': 160, 'colab': 150, 'local': 600}[PLATFORM]

## 3. Get the repo + CheckpointSync
Set `REPO_URL` to your AVA repo (git). We import the tested fabric rather than re-implement it.

In [ ]:
import sys, pathlib
REPO_URL = os.environ.get('AVA_REPO_URL', 'https://github.com/NAME0x0/AVA.git')  # TODO: your remote
ROOT = pathlib.Path('/kaggle/working' if PLATFORM=='kaggle' else '/content' if PLATFORM=='colab' else '.')
repo = ROOT / 'AVA'
if not repo.exists():
    # private repo: embed token in the clone URL for HTTPS auth
    url = REPO_URL.replace('https://', f'https://{tok}@') if 'github' in REPO_URL else REPO_URL
    os.system(f'git clone --depth 1 {url} {repo}')
scripts = repo / 'experiments' / 'exp6_v3' / 'scripts'
assert scripts.exists(), f'not found: {scripts} — fix REPO_URL'
sys.path.insert(0, str(scripts.parent))
from scripts.checkpoint_sync import CheckpointSync
print('CheckpointSync imported from', scripts)

## 4. Run config — the only cell you edit per phase

In [ ]:
CKPT_REPO = 'NAME0x0/AVA-v3-checkpoints'   # private Hub repo; auto-created
PHASE     = 'C1'                            # C1 baseline | C2 linearize | C5 sft | ...
DONOR     = 'Qwen/Qwen3.5-4B'               # swapped 2026-07-02 (REVIEW doc section 10)
SEED      = 1234
SEQ_LEN   = 1024
MICRO_BS  = 1
GRAD_ACC  = 16
LR        = 1e-4
SYNC_MIN  = 30                              # push cadence -> max work lost on preempt
TOTAL_STEPS = 20_000                        # phase length in optimizer steps
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 5. C1 — donor baseline (the bar every later phase must beat)
Light pass@1 on a handful of MBPP tasks, executed in-process. This is a smoke bar,
**not** the full EvalPlus/LiveCodeBench matrix — run those separately on Kaggle for the
real numbers. Skip this cell for training phases.

In [ ]:
import contextlib, io, signal
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load_donor_4bit(model_id):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    tok_ = AutoTokenizer.from_pretrained(model_id)
    m = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb,
                                             device_map='auto', torch_dtype=torch.bfloat16)
    return m, tok_

def run_code(src, timeout=5):
    g = {}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            exec(src, g)  # NOTE: notebook eval only. Real self-play uses the MCP sandbox.
        return True
    except Exception:
        return False

if PHASE == 'C1':
    from datasets import load_dataset
    model, tok_ = load_donor_4bit(DONOR)
    ds = load_dataset('mbpp', split='test[:20]')
    ok = 0
    for ex in ds:
        prompt = ex['text'] + '\n' + ex['test_list'][0] + '\n# solution:\n'
        ids = tok_(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**ids, max_new_tokens=256, do_sample=False)
        gen = tok_.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
        code_only = gen.split('```')[0]
        passed = run_code(code_only + '\n' + '\n'.join(ex['test_list']))
        ok += int(passed)
    print(f'DONOR {DONOR} MBPP[:20] rough pass@1 = {ok}/20 = {ok/20:.0%}')
    print('Record this as the C1 bar. Every transplant gate: no regression > 2pp vs this.')

## 6. Generic resumable LoRA shard trainer (C2+/C5)
Point `train_stream()` at the phase's dataset. Resumes from Hub, fast-forwards the
deterministic stream to the resumed step, trains until `SHARD_MINUTES`, force-saves, exits.
Re-run the notebook (any platform) to continue.

In [ ]:
import time
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

def build_trainable_donor(model_id):
    model, tok_ = load_donor_4bit(model_id)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                      task_type='CAUSAL_LM',
                      target_modules=['q_proj','k_proj','v_proj','o_proj',
                                      'gate_proj','up_proj','down_proj'])
    model = get_peft_model(model, lcfg)
    model.print_trainable_parameters()
    return model, tok_

def code_stream(tok_, seq_len):
    # C5 backbone. Swap for the-stack-v2 / CommitPackFT / teacher traces per phase.
    ds = load_dataset('nvidia/OpenCodeReasoning', 'split_0', split='train', streaming=True)
    for ex in ds:
        text = (ex.get('input','') or '') + '\n' + (ex.get('output','') or '')
        enc = tok_(text, truncation=True, max_length=seq_len, return_tensors='pt')
        if enc.input_ids.shape[1] < 8:
            continue
        yield enc.input_ids

def train():
    model, tok_ = build_trainable_donor(DONOR)
    opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)
    sync = CheckpointSync(CKPT_REPO, phase=PHASE, every_minutes=SYNC_MIN, trainable_only=True)
    start = sync.resume(model, opt)              # 0 if fresh; else next step
    print(f'starting at optimizer step {start}')

    stream = code_stream(tok_, SEQ_LEN)
    # fast-forward deterministic stream so resumed data == never-seen data
    consumed = start * GRAD_ACC
    for _ in range(consumed):
        next(stream, None)

    deadline = time.monotonic() + SHARD_MINUTES*60
    model.train()
    step = start
    while step < TOTAL_STEPS and time.monotonic() < deadline:
        opt.zero_grad(set_to_none=True)
        for _ in range(GRAD_ACC):
            ids = next(stream, None)
            if ids is None:
                stream = code_stream(tok_, SEQ_LEN)   # epoch wrap
                ids = next(stream)
            ids = ids.to(model.device)
            out = model(input_ids=ids, labels=ids)
            (out.loss / GRAD_ACC).backward()
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
        opt.step()
        if step % 20 == 0:
            print(f'step {step}  loss {out.loss.item():.3f}')
        sync.maybe_save(step, model, opt, extra={'loss': float(out.loss)})
        step += 1

    sync.save(step-1, model, opt, extra={'loss': float(out.loss), 'shard_end': True})
    print(f'shard done at step {step}. Re-run to continue (resumes from Hub).')

# Uncomment to run a training shard:
# train()

## 7. Resume anywhere
Nothing to configure. Re-open this notebook on **any** platform, run cells 0–4 and 6,
call `train()`. It pulls `LATEST.json` for `PHASE`, restores trainable weights + optimizer
+ RNG, fast-forwards the stream, and keeps going. Kaggle out of quota → switch to Colab →
switch to laptop overnight. Same repo, same run.

**Artifacts** (all in `NAME0x0/AVA-v3-checkpoints`):
`checkpoints/<phase>/step<N>/state.pt` + `checkpoints/<phase>/LATEST.json`.
At each phase gate, export the winning checkpoint to `NAME0x0/AVA-v3` (BF16) and later a GGUF build.